In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["font.size"] = 46
plt.rcParams["axes.labelsize"] = 58
plt.rcParams["axes.titlesize"] = 66
plt.rcParams["legend.fontsize"] = 46
plt.rcParams["xtick.labelsize"] = 44
plt.rcParams["ytick.labelsize"] = 44

# Create figure with 3 subplots (very tall + wide for full-width LaTeX)
fig, axes = plt.subplots(1, 3, figsize=(60, 22), dpi=200)

def exponential_moving_average(data, alpha=0.05):
    """Apply exponential moving average smoothing."""
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Define colors
colors = {
    "DART": "#2E86AB",      # Blue
    "PPO": "#A23B72",       # Purple/Pink
    "PPO Double Critic": "#F18F01"  # Orange
}

dart_line_color = "#333333"  # Dark gray for the vertical line
dart_line_style = {"color": dart_line_color, "linestyle": "--", "linewidth": 3.5, "alpha": 0.8, "zorder": 5}

# ========================
# Plot 1: MountainCar
# ========================
df_mountain = pd.read_csv("benchmark_results/mountaincar_ppo_dart.csv")

steps_dart = df_mountain["global_step"].values
dart_mean = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return"].values
dart_min = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return__MIN"].values
dart_max = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return__MAX"].values

ppo_mean = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return"].values
ppo_min = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return__MIN"].values
ppo_max = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps_dart = steps_dart[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

dart_ema = exponential_moving_average(dart_mean, alpha=0.05)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.05)

axes[0].plot(steps_dart, dart_ema, label="DART", color=colors["DART"], linewidth=5)
axes[0].fill_between(steps_dart, dart_min, dart_max, color=colors["DART"], alpha=0.2)

axes[0].plot(steps_dart, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5)
axes[0].fill_between(steps_dart, ppo_min, ppo_max, color=colors["PPO"], alpha=0.2)

axes[0].axvline(x=1.75e6, **dart_line_style)

axes[0].set_xlabel("Step", fontweight="bold")
axes[0].set_ylabel("Episodic Return", fontweight="bold")
axes[0].set_title("MountainCar-v0", fontweight="bold", pad=20)
axes[0].legend(loc="upper left", frameon=True, fancybox=True, shadow=True)
axes[0].grid(True, alpha=0.3)

# ========================
# Plot 2: Sparse Pendulum
# ========================
df_pendulum = pd.read_csv("benchmark_results/sparse_pendulum_ppo_dart_largeppo.csv")

steps_pend = df_pendulum["Step"].values
dart_mean = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return"].values
dart_min = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return__MIN"].values
dart_max = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return__MAX"].values

ppo_mean = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return"].values
ppo_min = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return__MIN"].values
ppo_max = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return__MAX"].values

ppo_large_mean = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return"].values
ppo_large_min = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return__MIN"].values
ppo_large_max = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean) & ~np.isnan(ppo_large_mean)
steps_pend = steps_pend[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]
ppo_large_mean, ppo_large_min, ppo_large_max = ppo_large_mean[mask], ppo_large_min[mask], ppo_large_max[mask]

dart_ema = exponential_moving_average(dart_mean, alpha=0.05)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.05)
ppo_large_ema = exponential_moving_average(ppo_large_mean, alpha=0.02)

axes[1].plot(steps_pend, dart_ema, label="DART", color=colors["DART"], linewidth=5)
axes[1].fill_between(steps_pend, dart_min, dart_max, color=colors["DART"], alpha=0.05)

axes[1].plot(steps_pend, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5)
axes[1].fill_between(steps_pend, ppo_min, ppo_max, color=colors["PPO"], alpha=0.05)

axes[1].plot(steps_pend, ppo_large_ema, label="PPO Double Critic", color=colors["PPO Double Critic"], linewidth=5)
axes[1].fill_between(steps_pend, ppo_large_min, ppo_large_max, color=colors["PPO Double Critic"], alpha=0.2)

axes[1].axvline(x=200, **dart_line_style)

axes[1].set_xlabel("Step", fontweight="bold")
axes[1].set_ylabel("Episodic Return", fontweight="bold")
axes[1].set_title("Sparse Pendulum", fontweight="bold", pad=20)
axes[1].legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
axes[1].grid(True, alpha=0.3)

# ========================
# Plot 3: Sparse Humanoid
# ========================
df_humanoid = pd.read_csv("benchmark_results/sparse_humanoid_ppo_dart_largeppo.csv")

steps_hum = df_humanoid["global_step"].values

# DART
dart_mean = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return"].values
dart_min = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return__MIN"].values
dart_max = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return__MAX"].values

# PPO
ppo_mean = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return"].values
ppo_min = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return__MIN"].values
ppo_max = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps_hum = steps_hum[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

dart_ema = exponential_moving_average(dart_mean, alpha=0.007)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.007)

axes[2].plot(steps_hum, dart_ema, label="DART", color=colors["DART"], linewidth=5.5)
axes[2].fill_between(steps_hum, dart_min, dart_max, color=colors["DART"], alpha=0.2)

axes[2].plot(steps_hum, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5.5)
axes[2].fill_between(steps_hum, ppo_min, ppo_max, color=colors["PPO"], alpha=0.2)

axes[2].axvline(x=2.8e7, **dart_line_style)

axes[2].set_xlabel("Step", fontweight="bold")
axes[2].set_ylabel("Episodic Return", fontweight="bold")
axes[2].set_title("Sparse Humanoid", fontweight="bold", pad=20)
axes[2].legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
axes[2].grid(True, alpha=0.3)

# Adjust layout and save
plt.tight_layout()
plt.savefig("dart_vs_ppo_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig("dart_vs_ppo_comparison.pdf", bbox_inches="tight")
print("Plots saved as 'dart_vs_ppo_comparison.png' and 'dart_vs_ppo_comparison.pdf'")
plt.show()

Plots saved as 'dart_vs_ppo_comparison.png' and 'dart_vs_ppo_comparison.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_13770/2057930297.py:163: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["font.size"] = 46
plt.rcParams["axes.labelsize"] = 58
plt.rcParams["axes.titlesize"] = 66
plt.rcParams["legend.fontsize"] = 46
plt.rcParams["xtick.labelsize"] = 44
plt.rcParams["ytick.labelsize"] = 44

# Create figure (single plot, tall + wide for full-width LaTeX)
fig, ax = plt.subplots(1, 1, figsize=(22, 18), dpi=200)

def exponential_moving_average(data, alpha=0.05):
    """Apply exponential moving average smoothing."""
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Define colors
colors = {
    "DART": "#2E86AB",      # Blue
    "PPO": "#A23B72",       # Purple/Pink
}

dart_line_color = "#333333"
dart_line_style = {"color": dart_line_color, "linestyle": "--", "linewidth": 3.5, "alpha": 0.8, "zorder": 5}

# ========================
# LLM PPO vs DART
# ========================
df = pd.read_csv("benchmark_results/LLM_ppo_dart_preliminary.csv")

steps = df["train/episode"].values

dart_mean = df["dart_enabled: true - train/objective/scores"].values
dart_min = df["dart_enabled: true - train/objective/scores__MIN"].values
dart_max = df["dart_enabled: true - train/objective/scores__MAX"].values

ppo_mean = df["dart_enabled: false - train/objective/scores"].values
ppo_min = df["dart_enabled: false - train/objective/scores__MIN"].values
ppo_max = df["dart_enabled: false - train/objective/scores__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps = steps[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

dart_ema = exponential_moving_average(dart_mean, alpha=0.05)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.05)

ax.plot(steps, dart_ema, label="DART", color=colors["DART"], linewidth=5)
ax.fill_between(steps, dart_min, dart_max, color=colors["DART"], alpha=0.2)

ax.plot(steps, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5)
ax.fill_between(steps, ppo_min, ppo_max, color=colors["PPO"], alpha=0.2)

ax.axvline(x=20000, **dart_line_style)

ax.set_xlabel("Episode", fontweight="bold")
ax.set_ylabel("Reward Score", fontweight="bold")
ax.legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("dart_vs_ppo_llm.png", dpi=300, bbox_inches="tight")
plt.savefig("dart_vs_ppo_llm.pdf", bbox_inches="tight")
print("Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.pdf'")
plt.show()

Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_13770/129839075.py:77: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
